# DramValue Whisky Price Dataset

**Secondary market whisky price intelligence data** scraped from 14 auction houses and retail sites.

This notebook demonstrates the dataset structure and provides exploratory analysis.

## Dataset Overview

| File | Rows | Description |
|------|------|-------------|
| `whisky_bottles.csv` | 9,642 | Bottle catalog with distillery, category, region, and price stats |
| `whisky_prices.csv` | 16,994 | Individual price records from auctions and retail |
| `whisky_market_stats.csv` | 1,719 | Monthly aggregate auction stats (2005-2023) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'sans-serif'

AMBER = '#d4a574'
AMBER_LIGHT = '#e8c9a0'
PEAT = '#1a1a1a'
GREEN = '#4ade80'
BLUE = '#60a5fa'

## 1. Load the Data

In [ ]:
bottles = pd.read_csv('whisky_bottles.csv')
prices = pd.read_csv('whisky_prices.csv', parse_dates=['transaction_date'])
market = pd.read_csv('whisky_market_stats.csv', parse_dates=['period_date'])

print(f'Bottles: {len(bottles):,} rows, {bottles.shape[1]} columns')
print(f'Prices:  {len(prices):,} rows, {prices.shape[1]} columns')
print(f'Market:  {len(market):,} rows, {market.shape[1]} columns')

In [ ]:
bottles.head()

In [ ]:
prices.head()

## 2. Category Distribution

In [ ]:
cat_counts = bottles['category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=AMBER, edgecolor='none')
ax.set_xlabel('Number of Bottles')
ax.set_title('Bottles by Category', fontsize=16, fontweight='bold', color='white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, cat_counts.values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10, color='white')

plt.tight_layout()
plt.show()

## 3. Price Distribution

In [ ]:
valid_prices = prices[prices['price_usd'].between(1, 50000)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid_prices['price_usd'], bins=100, color=AMBER, alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price Distribution (under $50K)', fontsize=14, fontweight='bold')
axes[0].axvline(valid_prices['price_usd'].median(), color=GREEN, linestyle='--', label=f'Median: ${valid_prices["price_usd"].median():,.0f}')
axes[0].legend()

# By source type
source_avg = valid_prices.groupby('source')['price_usd'].agg(['mean', 'median', 'count'])
source_avg = source_avg.sort_values('count', ascending=True)
axes[1].barh(source_avg.index, source_avg['median'], color=[AMBER, GREEN, BLUE][:len(source_avg)])
axes[1].set_xlabel('Median Price (USD)')
axes[1].set_title('Median Price by Source Type', fontsize=14, fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\nPrice Statistics:')
print(f'  Mean:   ${valid_prices["price_usd"].mean():,.2f}')
print(f'  Median: ${valid_prices["price_usd"].median():,.2f}')
print(f'  Max:    ${prices["price_usd"].max():,.2f}')

## 4. Market Trends Over Time

In [ ]:
monthly_agg = market.groupby('period_date').agg({
    'trading_volume': 'sum',
    'winning_bid_mean': 'mean',
    'lots_count': 'sum'
}).sort_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Trading volume
axes[0].fill_between(monthly_agg.index, monthly_agg['trading_volume'] / 1e6,
                     alpha=0.3, color=AMBER)
axes[0].plot(monthly_agg.index, monthly_agg['trading_volume'] / 1e6,
             color=AMBER, linewidth=1.5)
axes[0].set_ylabel('Volume ($M)')
axes[0].set_title('Monthly Whisky Auction Trading Volume', fontsize=14, fontweight='bold')

# Average winning bid
axes[1].plot(monthly_agg.index, monthly_agg['winning_bid_mean'],
             color=GREEN, linewidth=1.5)
axes[1].set_ylabel('Avg Winning Bid')
axes[1].set_title('Average Winning Bid Over Time', fontsize=14, fontweight='bold')

# Lots count
axes[2].bar(monthly_agg.index, monthly_agg['lots_count'] / 1000,
            width=25, color=BLUE, alpha=0.7)
axes[2].set_ylabel('Lots (thousands)')
axes[2].set_title('Monthly Lots Auctioned', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Date')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.1)

plt.tight_layout()
plt.show()

## 5. Top Auction Houses by Volume

In [ ]:
house_volume = market.groupby('auction_name')['trading_volume'].sum().sort_values(ascending=True)
top_houses = house_volume.tail(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_houses.index, top_houses.values / 1e6, color=AMBER)
ax.set_xlabel('Total Trading Volume ($M)')
ax.set_title('Top 10 Auction Houses by Trading Volume', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, top_houses.values / 1e6):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}M', va='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()

## 6. Price by Source

In [ ]:
source_prices = prices.groupby('source_name').agg(
    count=('price_usd', 'count'),
    mean_price=('price_usd', 'mean'),
    median_price=('price_usd', 'median')
).sort_values('count', ascending=False)

source_prices.style.format({
    'count': '{:,.0f}',
    'mean_price': '${:,.2f}',
    'median_price': '${:,.2f}'
})

## 7. Most Expensive Bottles

In [ ]:
top_bottles = bottles.nlargest(20, 'avg_price')[['name', 'distillery', 'category', 'avg_price', 'price_count']]
top_bottles.style.format({
    'avg_price': '${:,.2f}',
    'price_count': '{:,.0f}'
})

---

## About This Dataset

**Source**: [DramValue](https://github.com/tedrubin80/tracker) — an open-source whisky price intelligence platform.

**Collection Method**: Automated scraping of 14 auction houses and retail sites using Scrapy spiders with a 4-stage validation/normalization pipeline.

**Coverage**: 2017-2026, with market stats from 2005-2023.

**License**: Open data for research and analysis.